# Define a location and calculate the time series of diffuse radiation based on HOSTRADA/CERRA climate variables

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from hostrada4py import hostradaPoint as hp
import os
#os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "full"
os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "auto"

In [2]:
from hostrada4py import providerUI as provider_ui

provider_selector = provider_ui.create_provider_selector(
    globals(),
    initial="dwd",
    title="Select weather data provider",
    show=True,
)

def _sync_notebook_provider(change=None):
    supported = {value for _, value in provider_ui.variable_options(provider_dropdown.value)}

    current_variable = globals().get("HOSTRADA_VAR")
    if current_variable is not None and current_variable not in supported:
        globals()["HOSTRADA_VAR"] = "tas" if "tas" in supported else next(iter(supported))

    selector = globals().get("variable_selector")
    if selector is not None and hasattr(selector, "options"):
        all_options = globals().get("_hostrada_all_variable_options")
        if all_options is None:
            all_options = list(selector.options)
            globals()["_hostrada_all_variable_options"] = all_options
        filtered = [
            option for option in all_options
            if (option[1] if isinstance(option, tuple) else option) in supported
        ]
        old_values = tuple(value for value in selector.value if value in supported)
        selector.value = ()
        selector.options = filtered
        available = [option[1] if isinstance(option, tuple) else option for option in filtered]
        selector.value = old_values or (("tas",) if "tas" in available else tuple(available[:1]))

provider_dropdown.observe(_sync_notebook_provider, names="value")
_sync_notebook_provider()

## Definition of the location, the time period and download of the HOSTRADA/CERRA values

In [ ]:
lon = 13.32259
lat = 52.51712
start_UTC = "2025-08-01T00:00"
end_UTC = "2025-08-31T23:00"
df = hp.extract_diffuse_radiation_for_point(
    lon=lon, lat=lat, start=start_UTC, end=end_UTC,
    apply_weather_correction=False,
)
fn = "HOSTRADA_diffRad.csv"
df.to_csv(fn, index=False)
print(f"{len(df)} rows written to {fn}.")

## Time series of the calculated diffuse radiation

In [ ]:
df = pd.read_csv(fn)
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time")
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(df["time"], df["dhi"])
ax.set_xlabel("Date")
ax.set_ylabel("Diffuse radiation")
ax.set_title("Diffuse radiation in W/m2")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m %H:%M"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
plt.xticks(rotation=45)
plt.grid(True)
plt.show()